In [0]:
%sql
create or replace temp view tv_orders as
select value,
regexp_replace(value, '"order_date": (\\d{4}-\\d{2}-\\d{2})', '"order_date": "\$1"') as fixed_value
from gizmobox.bronze.v_orders;

In [0]:
%sql
select schema_of_json(fixed_value) as schema From tv_orders limit 1;




In [0]:
%sql
create table if not exists gizmobox.silver.orders_json
as
select
  from_json(
    fixed_value,
    'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>'
  ) as json_value
from
  tv_orders;

In [0]:
%sql
select * from  gizmobox.silver.orders_json;